In [2]:
spark

NameError: name 'spark' is not defined

In [ ]:
#Criação da estrutura da tabela
%SQL
create table if not exist bronze.ibge_municipios (
    id_ibge int COMMENT 'Chave primaria da tabela'
    id_municipio int COMMENT 'Id do municipio',
    nome_municipio str COMMENT 'Nome do municipio',
    id_microrregiao int COMMENT 'Id da microrregiao',
    nome_microrregiao str COMMENT 'Nome da microrregiao', 
    id_mesorregiao int COMMENT 'Id da mesorregiao', 
    nome_mesorregiao str COMMENT 'Nome da mesorregiao', 
    id_regiao_imediata int COMMENT 'Id da regiao imediata',
    nome_regiao_imediata str COMMENT 'Nome da regiao imediata',
    id_regiao_intermediaria int COMMENT 'Id da regiao intermediaria',
    nome_regiao_intermediaria str COMMENT 'Nome da regiao intermediaria',
    id_uf int COMMENT 'Id da Unidade Federal',
    sigla_uf str COMMENT 'Sigla da Unidade Federal',
    nome_uf str COMMENT 'Nome da Unidade Federal',
    id_regiao int COMMENT 'Id da regiao',
    sigla_regiao str COMMENT 'Sigla da regiao',
    nome_regiao str COMMENT 'Nome da Rregiao',
    cod_alteracao_registro str COMMENT 'Has criado para optimizar o processo. Utilizado na logica do merge para comparação de uma coluna so para atualizar o inserir um registro'
    tmp_inclusao timestamp COMMENT 'Data e hora da inclusão do registro'
    tmp_alteração timestamp COMMENT 'Data e hora da alteração do registro'
)
USING DELTA;

SyntaxError: expected 'else' after 'if' expression (2052817975.py, line 3)

In [ ]:
%sql

create or replace temporary view CTE_SOURCE as (
select
    id_municipio
    , nome_municipio
    , id_microrregiao
    , nome_microrregiao
    , id_mesorregiao
    , nome_mesorregiao
    , id_regiao_imediata
    , nome_regiao_imediata
    , id_regiao_intermediaria
    , nome_regiao_intermediaria
    , id_uf
    , sigla_uf
    , nome_uf
    , id_regiao
    , sigla_regiao
    , nome_regiao
from bronze.ibge_municipios
where tmp_inclusao > select(greatest(max(tmp_inclusao),max(tmp_alteracao)) from silver.ibge_municipios)
)

In [ ]:
%sql
create or replace temporary view CTE_CHAVE as (
select
    md5( cast(id_municipio as string ) as id_ibge
    , id_municipio
    , nome_municipio
    , id_microrregiao
    , nome_microrregiao
    , id_mesorregiao
    , nome_mesorregiao
    , id_regiao_imediata
    , nome_regiao_imediata
    , id_regiao_intermediaria
    , nome_regiao_intermediaria
    , id_uf
    , sigla_uf
    , nome_uf
    , id_regiao
    , sigla_regiao
    , nome_regiao
    , sha2(concat_ws('||', cast(id_municipio as string), nome_municipio, cast(id_microrregiao as string), 
                     nome_microrregiao, cast(id_mesorregiao as string) ,nome_mesorregiao, cast(id_regiao_imediata as string) 
                     , nome_regiao_imediata, cast(id_regiao_intermediaria as string), nome_regiao_intermediaria
                     , cast(id_uf as string), sigla_uf, nome_uf, cast(id_regiao as string), sigla_regiao, nome_regiao
                    )
        ) as cod_alteracao_registro
from CTE_SOURCE
    )

In [ ]:
MERGE INTO silver.ibge_municipios T using CTE_CHAVE S 
    on t.id_ibge = S.id_ibge
 
when matched and T.cod_alteracao_registro <> S.cod_alteracao_registro
    
    then
        T.id_municipio = S.id_municipio
        T.nome_municipio = S.nome_municipio
        T.id_microrregiao = S.id_microrregiao
        T.nome_microrregiao = S.nome_microrregiao
        T.id_mesorregiao = S.id_mesorregiao
        T.nome_mesorregiao = S.nome_mesorregiao
        T.id_regiao_imediata = S.id_regiao_imediata
        T.nome_regiao_imediata = S.nome_regiao_imediata
        T.id_regiao_intermediaria = S.id_regiao_intermediaria
        T.nome_regiao_intermediaria = S.nome_regiao_intermediaria
        T.id_uf = S.id_uf
        T.sigla_uf = S.sigla_uf
        T.nome_uf = S.nome_uf
        T.id_regiao = S.id_regiao
        T.sigla_regiao = S.sigla_regiao
        T.nome_regiao = S.nome_regiao
        T.cod_alteracao_registro = S.cod_alteracao_registro
        T.tmp_alteracao = current_timestamp()

WHEN NOT MATCHED THEN 
    insert
        T.id_municipio = S.id_municipio
        T.nome_municipio = S.nome_municipio
        T.id_microrregiao = S.id_microrregiao
        T.nome_microrregiao = S.nome_microrregiao
        T.id_mesorregiao = S.id_mesorregiao
        T.nome_mesorregiao = S.nome_mesorregiao
        T.id_regiao_imediata = S.id_regiao_imediata
        T.nome_regiao_imediata = S.nome_regiao_imediata
        T.id_regiao_intermediaria = S.id_regiao_intermediaria
        T.nome_regiao_intermediaria = S.nome_regiao_intermediaria
        T.id_uf = S.id_uf
        T.sigla_uf = S.sigla_uf
        T.nome_uf = S.nome_uf
        T.id_regiao = S.id_regiao
        T.sigla_regiao = S.sigla_regiao
        T.nome_regiao = S.nome_regiao
        T.cod_alteracao_registro = S.cod_alteracao_registro
        T.tmp_inclusao = current_timestamp()
        T.tmp_alteracao = cast(null as timestamp)
        



